# Naive Bayes Sentiment Analysis

## Setup

In this lab we will take advantage of some parts of the scikit-learn (sklearn) machine learning library to carry out a Naive Bayes
sentiment analysis of some amazon product reviews.

In addition to our usual libraries (numpy for linear algebra and bokeh for plotting) we load two functions
from sklearn.

- ```CountVectorizer``` extracts wordcounts from documents
- ```train_test_split``` splits our data up into a "training set" from which we will derive our probabilities, and 
    a "test" set that we will use to evaluate our classifier.

In [2]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import BernoulliNB
from sklearn.model_selection import train_test_split
from bokeh.models import ColumnDataSource, HoverTool
from bokeh.plotting import figure
from bokeh.io import show, output_notebook
import numpy as np
output_notebook()

Loading BokehJS ...

First we read our data from the file.  Each line of the file consists of a review, followed by a tab character,
followed by a "0" or "1".  We build a list of reviews and a corresponding list of labels from the file.

In [3]:
reviews = []
labels = []
with open("amazon.txt") as f:
    for line in f:
        review, label = line.strip().split('\t')
        reviews.append(review)
        labels.append(int(label))

The train_test_split breaks up the reviews and labels array randomly into two parts; by default, 75% of the data
goes into the train set and 25% into the test set, though this is adjustable. Now we set aside the test data
and work only with the training data until the end.

In [4]:
train_reviews, test_reviews, train_labels, test_labels = train_test_split(reviews, labels,random_state=11)
print('Length of train_reviews is ', len(train_reviews))
print('Length of test_reviews is ',len(test_reviews))

Length of train_reviews is  750
Length of test_reviews is  250


## A simple example

Now we use the CountVectorizer function to analyze our reviews.  The syntax here is that we create a CountVectorizer object and apply it to our review data.  We set options to say that  we only want to keep track of the 100 most common words in the reviews and (using ```binary=True```)that we only want to mark words that occur with a zero or 1 -- otherwise the routine will count the number of occurrences of each word.

In [5]:
F = CountVectorizer(max_features=100,binary=True)

To see how this works, let's apply it to a couple of simple sentences. The vectorizer expects a list of sentences, so we'll give a list of three sentences.  It returns a matrix whose rows correspond to the sentences and whose columns are features corresponding to the words it discovered in the data. 

In [6]:
simple = F.fit_transform(['This is a simple sentence that contains the word sentence twice.' ,
                          'This is another sentence.',
                          'A simple sentence has twice.'])
simple

<3x11 sparse matrix of type '<class 'numpy.int64'>'
	with 17 stored elements in Compressed Sparse Row format>

The vectorizer returns a "sparse" array, which is an efficient way to store large matrices which are mostly zero.  We'll
see how to view it in a moment.  

First, we can ask the vectorizer for the vocabulary that it uncovered.  It returns a python dictionary that associates
words to columns.  So for example in this case the word ```word``` corresponds to the 9th column of the data matrix.

In [7]:
F.vocabulary_

{'this': 8,
 'is': 3,
 'simple': 5,
 'sentence': 4,
 'that': 6,
 'contains': 1,
 'the': 7,
 'word': 10,
 'twice': 9,
 'another': 0,
 'has': 2}

Now the array ```simple```.

In [8]:
simple.toarray()

array([[0, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1],
       [1, 0, 0, 1, 1, 0, 0, 0, 1, 0, 0],
       [0, 0, 1, 0, 1, 1, 0, 0, 0, 1, 0]])

Each row is the feature vector to a sentence, and each column corresponds to a  word.  So the first sentence does *not* contain the first key word (```another```) but the second one does.

The column sums tell us how often each word occurs (total) in the documents.

In [9]:
simple.sum(axis=0)

matrix([[1, 1, 1, 2, 3, 2, 1, 1, 2, 2, 1]])

Suppose the first sentence is "positive" (labelled with 1) and the other two are negative (labelled with zero). The corresponding target array is Y.

In [10]:
Y = np.array([[1],[0],[0]])
Y

array([[1],
       [0],
       [0]])

We can compute the frequencies in the positive documents.

In [11]:
Y.transpose() @ simple


array([[0, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1]])

and in the negative ones.

In [12]:
(1-Y).transpose() @ simple

array([[1, 0, 1, 1, 2, 1, 0, 0, 1, 1, 0]])

The fourth word is 'sentence' which does indeed occur once in the type 1 sentences and 2 times in the type 0 ones. 

The numbers of sentences of each type are $Y^TY$ and $(1-Y)^T(1-Y)$ although numpy thinks these are two dimensional 1x1 arrays.

In [13]:
Y.transpose()@Y

array([[1]])

In [14]:
(1-Y).transpose()@(1-Y)

array([[2]])

We can compute the conditional probabilities with which each word occurs in the two types.  

In [15]:
Pplus = Y.transpose()@simple/(Y.transpose()@Y)
Pplus

array([[0., 1., 0., 1., 1., 1., 1., 1., 1., 1., 1.]])

Just as a check, the first word, ```another```, occurs only in the second sentence, so if has a 50% chance of
occurring in a sentence labelled zero.

In [16]:
Pminus = (1-Y).transpose()@simple/((1-Y).transpose()@(1-Y))
Pminus

array([[0.5, 0. , 0.5, 0.5, 1. , 0.5, 0. , 0. , 0.5, 0.5, 0. ]])

Now let's look at our training data.  First we use the CountVectorizer to compute the feature matrix.  We're going to add an option 
to tell the vectorizer to ignore elements of a list of "stop words" like he, his, at, him, ... to simplify things. 

## The product review data

In [17]:
vectorizer = CountVectorizer(max_features=100,binary=True,stop_words='english')

In [27]:
train_matrix = vectorizer.fit_transform(train_reviews).toarray()
keywords = vectorizer.get_feature_names_out()

In [28]:
train_matrix.shape

(750, 100)

Let's compute the frequencies of the 100 words in each of the two classes.  We convert our labels into a numpy array.

In [29]:
train_y = np.array(train_labels)

In [30]:
train_y.shape

(750,)

We use our formulae to compute the frequencies and the conditional probabilitiy vectors for the two classes.

In [31]:
freq_plus = (train_y.transpose()@train_matrix)
Nplus = train_y.transpose()@train_y
Pplus = freq_plus/Nplus
freq_minus = ((1-train_y).transpose()@train_matrix)
Nminus = ((1-train_y).transpose()@(1-train_y))
Pminus = freq_minus/Nminus
N = Nminus+Nplus

In [32]:
print(Nplus, Nminus)

376 374


Here we use a trick called "indirect sort" to find the words with largest P(w|+) and P(w|-).  Argsort returns this *locations* of
the elements in order.  So indices[0] is the location in the Pplus array with the smallest value, indices[1] the next smallest value, and so on.
We use these indices to extract the corresponding keywords.

In [33]:
indices = np.argsort(Pplus)
[keywords[i] for i in indices[::-1]][:20]

['great',
 'phone',
 'good',
 'works',
 'headset',
 'sound',
 'excellent',
 'quality',
 'product',
 'price',
 'use',
 'love',
 'battery',
 'recommend',
 'nice',
 'best',
 'easy',
 'case',
 'really',
 'like']

In [34]:
indices = np.argsort(Pminus)
[keywords[i] for i in indices[::-1]][:20]

['phone',
 'work',
 'battery',
 'money',
 'product',
 'don',
 'does',
 'waste',
 'time',
 'headset',
 'quality',
 'ear',
 'use',
 'bad',
 'worst',
 'sound',
 'piece',
 'good',
 'service',
 'calls']

In [35]:
# This is a fancy use of bokeh to add hover labels to the dots
source = ColumnDataSource({'+':Pplus,'-':Pminus,'word':keywords})
f=figure()
f.scatter(x='+',y='-',source=source)

f.xaxis.axis_label='P(w|+)'
f.yaxis.axis_label = 'P(w|-)'
f.line(x=[0,.2],y=[0,.2])
f.add_tools(HoverTool(tooltips=[("word","@word")]))
show(f)

To avoid taking the logarithm of zero, we increase all of the frequency counts by 1, as well as the Nplus and Nminus by 1. This is often called
"smoothing."


In [36]:
freq_plus = freq_plus+1
freq_minus = freq_minus+1
Nplus = Nplus+2
Nminus = Nminus+2

In [37]:
LPplus = np.log(freq_plus/Nplus)
LPNplus = np.log(1-freq_plus/Nplus)
LPminus = np.log(freq_minus/Nminus)
LPNminus = np.log(1-freq_minus/Nminus)

Using the equation from the notes (which is essentially Bayes rule), we find:

In [38]:
posL = train_matrix @ LPplus + (1-train_matrix) @(LPNplus) - np.log(Nplus/(N+2))
negL  = train_matrix @ LPminus + (1-train_matrix)@(LPNminus) - np.log(Nminus/(N+2))

Recall that posL and negL are the likelihoods that a particular review is positive or negative, and our decision criterion is:
- label 1 if posL-negL>0
- label 0 otherwise

Our decision array has a 1 if posL>negL and a zero otherwise.

In [39]:
decision = (posL > negL).astype(int)

Our check array as a 1 if decision and the original label agree, and zero otherwise.

In [40]:
check = (decision==train_y).astype(int)

In [41]:
np.sum(check)

591

They agree 591/750 times, or about 75% of the time.  That's much better than guessing, which would only be right 50% of the time.

Finally, we use the test data to see if we can predict labels on "new" data.
We re-use the LPplus and LPminus parameters, as well as the Nplus/N and Nminus/N from the training data.
But we need to compute the data matrix for the test data *based on the features derived from the training data.*

In [42]:
vectorizer.fit(train_reviews)
test_matrix = vectorizer.transform(test_reviews).toarray()
test_y = np.array(test_labels)

In [43]:
test_matrix.shape

(250, 100)

In [44]:
posL = test_matrix @ LPplus + (1-test_matrix)@(LPNplus) -np.log(Nplus/(Nplus+Nminus))
negL = test_matrix @ LPminus + (1-test_matrix)@(LPNminus) - np.log(Nminus/(Nplus+Nminus))

In [45]:
test_decision = (posL > negL).astype(int)
check = (test_decision == test_y).astype(int)
np.sum(check)

182

So we have correctly classified 182/250 reviews from the test set for an accuracy of 171/250 = 73%. Much better than guessing!

## Using the sklearn facilities

The sklearn library can do all of this using built in routines.  We add to the work above one more import.

In [1]:
from sklearn.naive_bayes import BernoulliNB, MultinomialNB

The BernoulliNB function takes the binary feature matrix and does all the computations associated with fitting the naive bayes
model.  Let's walk through it. We start with the train_reviews and train_labels lists.  This cell builds the Bernoulli classifier B
using the vectorized train_reviews and the train_labels data.

In [ ]:
vectorizer = CountVectorizer(max_features=100,binary=True,stop_words='english')
V = vectorizer.fit(train_reviews)
X = V.transform(train_reviews)
train_y = np.array(train_labels)
B = BernoulliNB().fit(X,train_y)

This cell uses the fitted vectorizer to compute the data matrix for the test reviews.

In [88]:
X.toarray()[0,:]

array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0])

In [89]:
T = V.transform(test_reviews)
test_y = np.array(test_labels)

Now we find the predictions using the matrix T.

In [90]:
predictions = B.predict(T)

In [91]:
predictions

array([1, 1, 0, 1, 1, 1, 0, 1, 1, 1, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 1,
       0, 0, 0, 0, 1, 1, 0, 0, 1, 1, 0, 0, 0, 1, 1, 0, 0, 0, 1, 1, 0, 0,
       0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 1, 1, 0, 0, 0, 1, 0, 0, 1, 1, 0, 0, 1, 1, 0, 0, 1, 1, 1, 1, 0,
       1, 0, 0, 0, 0, 1, 0, 0, 1, 1, 0, 1, 1, 1, 0, 0, 1, 1, 1, 0, 0, 0,
       1, 1, 0, 1, 1, 1, 0, 1, 0, 0, 1, 0, 0, 0, 1, 0, 0, 1, 1, 1, 1, 0,
       1, 1, 1, 0, 0, 1, 1, 1, 0, 1, 0, 0, 0, 0, 1, 0, 0, 1, 1, 1, 0, 0,
       1, 0, 1, 1, 0, 1, 1, 0, 0, 0, 1, 1, 1, 0, 0, 0, 1, 0, 0, 1, 0, 0,
       0, 0, 1, 0, 0, 0, 1, 1, 0, 0, 0, 0, 1, 1, 1, 0, 0, 0, 0, 0, 1, 0,
       0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 1, 1, 1, 0, 1, 0, 0, 0,
       0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 1, 0, 1, 0, 0, 0, 0, 1, 0, 0,
       0, 0, 0, 1, 1, 1, 0, 0])

In [93]:
test_y

array([1, 1, 1, 0, 1, 1, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 1,
       0, 1, 0, 1, 1, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0,
       1, 1, 0, 0, 0, 0, 1, 1, 0, 0, 0, 1, 0, 0, 0, 0, 1, 1, 1, 1, 0, 0,
       0, 0, 0, 0, 0, 1, 1, 1, 0, 1, 1, 1, 0, 1, 0, 0, 1, 0, 1, 0, 1, 1,
       1, 1, 0, 0, 1, 1, 0, 0, 1, 0, 0, 1, 1, 0, 1, 0, 1, 0, 1, 0, 0, 0,
       1, 0, 0, 1, 1, 1, 0, 1, 0, 1, 1, 0, 1, 0, 0, 0, 1, 1, 1, 1, 1, 0,
       1, 1, 0, 1, 0, 1, 1, 1, 1, 1, 1, 0, 0, 1, 1, 1, 1, 0, 1, 1, 0, 0,
       0, 0, 1, 1, 0, 1, 1, 1, 0, 0, 1, 1, 1, 0, 1, 0, 1, 1, 0, 1, 0, 0,
       0, 0, 1, 0, 0, 1, 1, 1, 0, 1, 1, 0, 1, 1, 1, 0, 1, 0, 0, 0, 1, 0,
       0, 0, 0, 1, 1, 1, 1, 0, 0, 0, 1, 0, 0, 1, 1, 1, 0, 0, 1, 1, 0, 0,
       1, 0, 0, 1, 1, 1, 0, 1, 0, 1, 0, 0, 1, 0, 0, 1, 1, 0, 0, 1, 0, 1,
       1, 0, 1, 1, 0, 1, 1, 0])

The score method allows us to tell how well we did.

In [94]:
B.score(T,test_y)

0.676

The logs of the probabilities P(w|+/-) are stored inside B, and they agree with our computations.

In [96]:
B.feature_log_prob_.shape

(2, 100)

In [103]:
B.feature_log_prob_[:,1]

array([-5.26009615, -3.8313551 ])

In [104]:
B.score(T,test_y)

0.676

In [ ]:
LPminus

In [ ]:
LPplus

## Your turn

Carry out the analysis above using the yelp and imdb data.  You can use the sklearn facilities to make your life easier if you want.
You can also try the multinomial classifier to see if it works better.  In that case, you need to remove the binary=True flag
from the vectorizer so that it counts frequencies.  Here is an example of how the countvectorizer can compute term frequencies. You can also experiment with the "stop_words" flag.

In [46]:
vectorizer = CountVectorizer(max_features=10)

In [47]:
V = vectorizer.fit(["Here is a sentence", "Here is another sentence"])

In [48]:
V.vocabulary_

{'here': 1, 'is': 2, 'sentence': 3, 'another': 0}

In [49]:
X = V.transform(["What are the frequencies in this here sentence", "I wrote a sentence about this sentences"])

In [50]:
X.toarray()

array([[0, 1, 0, 1],
       [0, 0, 0, 1]])

In [52]:
V.get_feature_names_out()

array(['another', 'here', 'is', 'sentence'], dtype=object)

In [53]:
reviews = []
labels = []
with open("yelp.txt") as f:
    for line in f:
        review, label = line.strip().split('\t')
        reviews.append(review)
        labels.append(int(label))

In [54]:
train_reviews, test_reviews, train_labels, test_labels = train_test_split(reviews, labels,random_state=11)
print('Length of train_reviews is ', len(train_reviews))
print('Length of test_reviews is ',len(test_reviews))

Length of train_reviews is  750
Length of test_reviews is  250


In [55]:
vectorizer = CountVectorizer(max_features=100,binary=True,stop_words='english')
V = vectorizer.fit(train_reviews)
X = V.transform(train_reviews)
train_y = np.array(train_labels)
B = BernoulliNB().fit(X,train_y)

In [56]:
T = V.transform(test_reviews)
test_y = np.array(test_labels)

In [57]:
B.predict(T)

array([1, 1, 0, 1, 1, 1, 0, 1, 1, 1, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 1,
       0, 0, 0, 0, 1, 1, 0, 0, 1, 1, 0, 0, 0, 1, 1, 0, 0, 0, 1, 1, 0, 0,
       0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 1, 1, 0, 0, 0, 1, 0, 0, 1, 1, 0, 0, 1, 1, 0, 0, 1, 1, 1, 1, 0,
       1, 0, 0, 0, 0, 1, 0, 0, 1, 1, 0, 1, 1, 1, 0, 0, 1, 1, 1, 0, 0, 0,
       1, 1, 0, 1, 1, 1, 0, 1, 0, 0, 1, 0, 0, 0, 1, 0, 0, 1, 1, 1, 1, 0,
       1, 1, 1, 0, 0, 1, 1, 1, 0, 1, 0, 0, 0, 0, 1, 0, 0, 1, 1, 1, 0, 0,
       1, 0, 1, 1, 0, 1, 1, 0, 0, 0, 1, 1, 1, 0, 0, 0, 1, 0, 0, 1, 0, 0,
       0, 0, 1, 0, 0, 0, 1, 1, 0, 0, 0, 0, 1, 1, 1, 0, 0, 0, 0, 0, 1, 0,
       0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 1, 1, 1, 0, 1, 0, 0, 0,
       0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 1, 0, 1, 0, 0, 0, 0, 1, 0, 0,
       0, 0, 0, 1, 1, 1, 0, 0])

In [58]:
test_y

array([1, 1, 1, 0, 1, 1, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 1,
       0, 1, 0, 1, 1, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0,
       1, 1, 0, 0, 0, 0, 1, 1, 0, 0, 0, 1, 0, 0, 0, 0, 1, 1, 1, 1, 0, 0,
       0, 0, 0, 0, 0, 1, 1, 1, 0, 1, 1, 1, 0, 1, 0, 0, 1, 0, 1, 0, 1, 1,
       1, 1, 0, 0, 1, 1, 0, 0, 1, 0, 0, 1, 1, 0, 1, 0, 1, 0, 1, 0, 0, 0,
       1, 0, 0, 1, 1, 1, 0, 1, 0, 1, 1, 0, 1, 0, 0, 0, 1, 1, 1, 1, 1, 0,
       1, 1, 0, 1, 0, 1, 1, 1, 1, 1, 1, 0, 0, 1, 1, 1, 1, 0, 1, 1, 0, 0,
       0, 0, 1, 1, 0, 1, 1, 1, 0, 0, 1, 1, 1, 0, 1, 0, 1, 1, 0, 1, 0, 0,
       0, 0, 1, 0, 0, 1, 1, 1, 0, 1, 1, 0, 1, 1, 1, 0, 1, 0, 0, 0, 1, 0,
       0, 0, 0, 1, 1, 1, 1, 0, 0, 0, 1, 0, 0, 1, 1, 1, 0, 0, 1, 1, 0, 0,
       1, 0, 0, 1, 1, 1, 0, 1, 0, 1, 0, 0, 1, 0, 0, 1, 1, 0, 0, 1, 0, 1,
       1, 0, 1, 1, 0, 1, 1, 0])

In [59]:
T

<250x100 sparse matrix of type '<class 'numpy.int64'>'
	with 388 stored elements in Compressed Sparse Row format>

In [66]:
review = "This restaurant is great."
V.transform([review])

<1x100 sparse matrix of type '<class 'numpy.int64'>'
	with 2 stored elements in Compressed Sparse Row format>

In [67]:
V.get_feature_names_out()

array(['amazing', 'atmosphere', 'awesome', 'bad', 'best', 'better', 'bit',
       'bland', 'breakfast', 'buffet', 'burger', 'came', 'chicken',
       'clean', 'cold', 'come', 'coming', 'day', 'definitely',
       'delicious', 'did', 'didn', 'disappointed', 'dish', 'don', 'eat',
       'eating', 'excellent', 'experience', 'fantastic', 'feel', 'felt',
       'flavor', 'food', 'fresh', 'friendly', 'fries', 'going', 'good',
       'got', 'great', 'happy', 'just', 'know', 'like', 'little', 'll',
       'love', 'loved', 'make', 'meal', 'menu', 'minutes', 'nice',
       'night', 'order', 'ordered', 'overall', 'pizza', 'place', 'poor',
       'pretty', 'prices', 'quality', 'really', 'recommend', 'restaurant',
       'right', 'salad', 'sandwich', 'sauce', 'say', 'server', 'service',
       'slow', 'soon', 'spicy', 'staff', 'stars', 'steak', 'super',
       'sure', 'sushi', 'tasted', 'terrible', 'thing', 'think', 'time',
       'times', 've', 'vegas', 'wait', 'waitress', 'want', 'wasn', 'way',
 

In [68]:
B.predict(V.transform([review]))

array([1])

In [75]:
V.vocabulary_['terrible']

84

In [77]:
B.feature_log_prob_[0,84]

-3.756018756951565

In [78]:
V.vocabulary_['great']

40

In [80]:
B.feature_log_prob_[0,40]

-5.953243334287785

In [105]:
# Sort words by positive-class log probability (class label 1).
pos_index = int(np.where(B.classes_ == 1)[0][0])
words = V.get_feature_names_out()
pos_log = B.feature_log_prob_[pos_index]
sorted_idx = np.argsort(pos_log)[::-1]
pos_words_sorted = words[sorted_idx]
pos_log_sorted = pos_log[sorted_idx]
list(zip(pos_words_sorted[:50], pos_log_sorted[:50]))

[('good', -1.9789710113162013),
 ('food', -2.0606490423304686),
 ('great', -2.1041341542702074),
 ('place', -2.1731270257571587),
 ('service', -2.197224577336219),
 ('friendly', -2.819754190682211),
 ('delicious', -2.966357664874087),
 ('really', -3.0204248861443626),
 ('just', -3.0204248861443626),
 ('nice', -3.077583299984311),
 ('time', -3.077583299984311),
 ('amazing', -3.202746442938317),
 ('best', -3.2717393144252687),
 ('staff', -3.3458472865789903),
 ('like', -3.3458472865789903),
 ('vegas', -3.4258899942525267),
 ('experience', -3.4258899942525267),
 ('love', -3.4258899942525267),
 ('restaurant', -3.4258899942525267),
 ('pretty', -3.5129013712421564),
 ('fantastic', -3.5129013712421564),
 ('loved', -3.608211551046481),
 ('awesome', -3.608211551046481),
 ('menu', -3.608211551046481),
 ('server', -3.608211551046481),
 ('prices', -3.7135720667043075),
 ('definitely', -3.7135720667043075),
 ('atmosphere', -3.8313551023606913),
 ('fresh', -3.8313551023606913),
 ('excellent', -3.831

## IMDB Movie Reviews Analysis

Now let's apply the same Naive Bayes approach to the IMDB movie reviews dataset.

In [107]:
# Load IMDB data
imdb_reviews = []
imdb_labels = []

with open('imdb.txt','r') as f:
    for line in f:
        review, label = line.rsplit('\t',1)  # Split from the right to handle tabs in text
        imdb_reviews.append(review.strip())
        imdb_labels.append(int(label.strip()))

print(f"Loaded {len(imdb_reviews)} IMDB reviews")

Loaded 1000 IMDB reviews


In [116]:
# Split into training and test sets
from sklearn.model_selection import train_test_split

imdb_train_reviews, imdb_test_reviews, imdb_train_labels, imdb_test_labels = train_test_split(
    imdb_reviews, imdb_labels, random_state=42, test_size=0.25
)

print(f"Training set: {len(imdb_train_reviews)} reviews")
print(f"Test set: {len(imdb_test_reviews)} reviews")

Training set: 750 reviews
Test set: 250 reviews


In [117]:
# Create vectorizer for IMDB data
from sklearn.feature_extraction.text import CountVectorizer

imdb_vectorizer = CountVectorizer(binary=True, max_features=1000, stop_words='english')
imdb_X_train = imdb_vectorizer.fit_transform(imdb_train_reviews)
imdb_X_test = imdb_vectorizer.transform(imdb_test_reviews)

print(f"Vocabulary size: {len(imdb_vectorizer.get_feature_names_out())}")
print(f"Training matrix shape: {imdb_X_train.shape}")

Vocabulary size: 1000
Training matrix shape: (750, 1000)


In [118]:
# Convert labels to numpy arrays
imdb_train_y = np.array(imdb_train_labels)
imdb_test_y = np.array(imdb_test_labels)

In [119]:
# Train Bernoulli Naive Bayes classifier on IMDB data
imdb_classifier = BernoulliNB().fit(imdb_X_train, imdb_train_y)
print("Classifier trained successfully")

Classifier trained successfully


In [120]:
# Evaluate training accuracy
imdb_train_predictions = imdb_classifier.predict(imdb_X_train)
imdb_train_accuracy = (imdb_train_predictions == imdb_train_y).mean()
print(f"Training accuracy: {imdb_train_accuracy:.4f}")

Training accuracy: 0.9293


In [121]:
# Evaluate test accuracy
imdb_test_predictions = imdb_classifier.predict(imdb_X_test)
imdb_test_accuracy = (imdb_test_predictions == imdb_test_y).mean()
print(f"Test accuracy: {imdb_test_accuracy:.4f}")

Test accuracy: 0.7520


### Most Informative Words

Let's identify which words are most strongly associated with positive vs. negative reviews.

In [122]:
# Get feature names and log probabilities
imdb_words = imdb_vectorizer.get_feature_names_out()

# Find which class is 0 (negative) and which is 1 (positive)
imdb_neg_idx = int(np.where(imdb_classifier.classes_ == 0)[0][0])
imdb_pos_idx = int(np.where(imdb_classifier.classes_ == 1)[0][0])

imdb_neg_log_prob = imdb_classifier.feature_log_prob_[imdb_neg_idx]
imdb_pos_log_prob = imdb_classifier.feature_log_prob_[imdb_pos_idx]

In [123]:
# Top 30 words most associated with POSITIVE reviews (highest log prob for class 1)
imdb_pos_sorted_idx = np.argsort(imdb_pos_log_prob)[::-1]
print("Top 30 words for POSITIVE reviews:")
for i in range(30):
    idx = imdb_pos_sorted_idx[i]
    print(f"{imdb_words[idx]:15s} {imdb_pos_log_prob[idx]:.4f}")

Top 30 words for POSITIVE reviews:
film            -1.6014
movie           -1.7600
good            -2.3925
great           -2.6231
best            -3.0285
just            -3.0285
love            -3.0285
wonderful       -3.1463
like            -3.2108
acting          -3.2798
excellent       -3.2798
really          -3.3539
characters      -3.3539
character       -3.3539
10              -3.4340
think           -3.5210
actors          -3.5210
right           -3.5210
look            -3.5210
time            -3.5210
beautiful       -3.5210
cast            -3.5210
films           -3.5210
music           -3.6163
funny           -3.6163
performance     -3.6163
art             -3.7217
story           -3.7217
play            -3.7217
actor           -3.7217


In [124]:
# Top 30 words most associated with NEGATIVE reviews (highest log prob for class 0)
imdb_neg_sorted_idx = np.argsort(imdb_neg_log_prob)[::-1]
print("Top 30 words for NEGATIVE reviews:")
for i in range(30):
    idx = imdb_neg_sorted_idx[i]
    print(f"{imdb_words[idx]:15s} {imdb_neg_log_prob[idx]:.4f}")

Top 30 words for NEGATIVE reviews:
movie           -1.6688
film            -2.0742
bad             -2.1388
just            -2.4489
acting          -2.9009
plot            -2.9497
time            -2.9497
like            -3.1122
didn            -3.1122
script          -3.1728
good            -3.1728
don             -3.2374
story           -3.2374
stupid          -3.3064
really          -3.3064
awful           -3.3805
work            -3.4605
terrible        -3.4605
make            -3.5475
scenes          -3.6428
way             -3.6428
waste           -3.6428
worst           -3.6428
characters      -3.6428
seen            -3.6428
look            -3.7482
watching        -3.7482
real            -3.7482
thing           -3.7482
worse           -3.7482


In [125]:
# Most discriminative words (ratio of log probabilities)
# Positive ratio > 0 means word favors positive reviews
# Negative ratio < 0 means word favors negative reviews
imdb_log_ratio = imdb_pos_log_prob - imdb_neg_log_prob

# Top words favoring positive reviews
print("\nTop 20 words favoring POSITIVE reviews (highest log ratio):")
pos_ratio_sorted = np.argsort(imdb_log_ratio)[::-1]
for i in range(20):
    idx = pos_ratio_sorted[i]
    print(f"{imdb_words[idx]:15s} ratio: {imdb_log_ratio[idx]:.4f}")


Top 20 words favoring POSITIVE reviews (highest log ratio):
love            ratio: 2.9169
wonderful       ratio: 2.7991
excellent       ratio: 2.6656
beautiful       ratio: 2.4244
great           ratio: 2.2238
played          ratio: 2.1060
loved           ratio: 2.1060
enjoyed         ratio: 2.1060
job             ratio: 2.1060
actually        ratio: 1.9724
incredible      ratio: 1.9724
liked           ratio: 1.9724
hilarious       ratio: 1.8183
history         ratio: 1.8183
brilliant       ratio: 1.8183
cinema          ratio: 1.8183
portrayal       ratio: 1.8183
nice            ratio: 1.8183
cool            ratio: 1.8183
subtle          ratio: 1.8183


In [126]:
# Top words favoring negative reviews
print("\nTop 20 words favoring NEGATIVE reviews (lowest log ratio):")
neg_ratio_sorted = np.argsort(imdb_log_ratio)
for i in range(20):
    idx = neg_ratio_sorted[i]
    print(f"{imdb_words[idx]:15s} ratio: {imdb_log_ratio[idx]:.4f}")


Top 20 words favoring NEGATIVE reviews (lowest log ratio):
bad             ratio: -2.6815
stupid          ratio: -2.6125
awful           ratio: -2.5384
terrible        ratio: -2.4584
worst           ratio: -2.2761
waste           ratio: -2.2761
plot            ratio: -2.2761
sucked          ratio: -1.9194
half            ratio: -1.9194
avoid           ratio: -1.7652
action          ratio: -1.7652
holes           ratio: -1.7652
crap            ratio: -1.7652
slow            ratio: -1.7652
lacks           ratio: -1.7652
mess            ratio: -1.7652
script          ratio: -1.6474
hours           ratio: -1.5829
poor            ratio: -1.5829
hour            ratio: -1.5829


### Test the Classifier

Let's test the classifier on a few sample movie reviews.

In [127]:
# Test on sample reviews
test_samples = [
    "This movie was absolutely brilliant and amazing!",
    "Terrible waste of time. The acting was awful.",
    "Great performances and beautiful cinematography.",
    "Boring and predictable plot with bad dialogue."
]

for sample in test_samples:
    sample_vec = imdb_vectorizer.transform([sample])
    prediction = imdb_classifier.predict(sample_vec)[0]
    sentiment = "POSITIVE" if prediction == 1 else "NEGATIVE"
    print(f"{sentiment}: {sample}")

POSITIVE: This movie was absolutely brilliant and amazing!
NEGATIVE: Terrible waste of time. The acting was awful.
POSITIVE: Great performances and beautiful cinematography.
NEGATIVE: Boring and predictable plot with bad dialogue.


## Sentiment Analysis Results

Comprehensive evaluation of the sentiment classifier on IMDB reviews.

In [ ]:
# Sentiment distribution in dataset
print("\nSentiment Distribution in IMDB Dataset:")
print(f"Total reviews: {len(imdb_labels)}")
print(f"Positive reviews (1): {sum(imdb_labels)} ({100*sum(imdb_labels)/len(imdb_labels):.1f}%)")
print(f"Negative reviews (0): {len(imdb_labels)-sum(imdb_labels)} ({100*(len(imdb_labels)-sum(imdb_labels))/len(imdb_labels):.1f}%)")

In [128]:
# Confusion Matrix and Classification Metrics
from sklearn.metrics import confusion_matrix, precision_score, recall_score, f1_score, accuracy_score

# Compute confusion matrix
cm = confusion_matrix(imdb_test_y, imdb_test_predictions)

# Classification metrics
accuracy = accuracy_score(imdb_test_y, imdb_test_predictions)
precision = precision_score(imdb_test_y, imdb_test_predictions)
recall = recall_score(imdb_test_y, imdb_test_predictions)
f1 = f1_score(imdb_test_y, imdb_test_predictions)

print("\nConfusion Matrix (Test Set):")
print(f"              Predicted Negative  Predicted Positive")
print(f"Actual Neg:   {cm[0,0]:6d}                {cm[0,1]:6d}")
print(f"Actual Pos:   {cm[1,0]:6d}                {cm[1,1]:6d}")

print(f"\nClassification Metrics (Test Set):")
print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1-Score:  {f1:.4f}")


Confusion Matrix (Test Set):
              Predicted Negative  Predicted Positive
Actual Neg:      103                    17
Actual Pos:       45                    85

Classification Metrics (Test Set):
Accuracy:  0.7520
Precision: 0.8333
Recall:    0.6538
F1-Score:  0.7328


In [129]:
# Error Analysis - Find misclassified examples
misclassified_idx = np.where(imdb_test_predictions != imdb_test_y)[0]

print(f"\nTotal misclassified reviews: {len(misclassified_idx)} out of {len(imdb_test_y)}")

# False positives (predicted positive, actually negative)
false_pos = misclassified_idx[(imdb_test_y[misclassified_idx] == 0) & (imdb_test_predictions[misclassified_idx] == 1)]
# False negatives (predicted negative, actually positive)
false_neg = misclassified_idx[(imdb_test_y[misclassified_idx] == 1) & (imdb_test_predictions[misclassified_idx] == 0)]

print(f"\nFalse Positives (predicted positive, actually negative): {len(false_pos)}")
print(f"False Negatives (predicted negative, actually positive): {len(false_neg)}")

# Show a few examples
print("\n--- Examples of False Positives (incorrectly classified as positive) ---")
for i in range(min(2, len(false_pos))):
    idx = false_pos[i]
    print(f"\nActual: NEGATIVE | Predicted: POSITIVE")
    print(f"{imdb_test_reviews[idx][:200]}...")


Total misclassified reviews: 62 out of 250

False Positives (predicted positive, actually negative): 17
False Negatives (predicted negative, actually positive): 45

--- Examples of False Positives (incorrectly classified as positive) ---

Actual: NEGATIVE | Predicted: POSITIVE
This film tries to be a serious and sophisticated thriller/horror flick and it fails miserably....

Actual: NEGATIVE | Predicted: POSITIVE
There are the usual Hitchcock logic flaws....


In [130]:
print("\n--- Examples of False Negatives (incorrectly classified as negative) ---")
for i in range(min(2, len(false_neg))):
    idx = false_neg[i]
    print(f"\nActual: POSITIVE | Predicted: NEGATIVE")
    print(f"{imdb_test_reviews[idx][:200]}...")


--- Examples of False Negatives (incorrectly classified as negative) ---

Actual: POSITIVE | Predicted: NEGATIVE
This is a witty and delightful adaptation of the Dr Seuss book, brilliantly animated by UPA's finest and thoroughly deserving of its Academy Award....

Actual: POSITIVE | Predicted: NEGATIVE
I especially liked the non-cliche choices with the parents; in other movies, I could predict the dialog verbatim, but the writing in this movie made better selections....


In [131]:
# Prediction Probabilities
imdb_pred_proba = imdb_classifier.predict_proba(imdb_X_test)

# Find predictions with lowest confidence
max_proba = np.max(imdb_pred_proba, axis=1)
low_confidence_idx = np.argsort(max_proba)[:5]

print("\n--- Low Confidence Predictions ---")
print("Reviews where classifier was least certain:")

for i, idx in enumerate(low_confidence_idx):
    neg_prob = imdb_pred_proba[idx, 0]
    pos_prob = imdb_pred_proba[idx, 1]
    prediction = "POSITIVE" if imdb_test_predictions[idx] == 1 else "NEGATIVE"
    actual = "POSITIVE" if imdb_test_y[idx] == 1 else "NEGATIVE"
    confidence = max(neg_prob, pos_prob)
    print(f"\n{i+1}. Prediction: {prediction} ({confidence:.2%}) | Actual: {actual}")
    print(f"   Negative prob: {neg_prob:.4f}, Positive prob: {pos_prob:.4f}")
    print(f"   {imdb_test_reviews[idx][:150]}...")


--- Low Confidence Predictions ---
Reviews where classifier was least certain:

1. Prediction: POSITIVE (51.21%) | Actual: POSITIVE
   Negative prob: 0.4879, Positive prob: 0.5121
   The scenes are often funny and occasionally touching as the characters evaluate their lives and where they are going....

2. Prediction: POSITIVE (51.31%) | Actual: NEGATIVE
   Negative prob: 0.4869, Positive prob: 0.5131
   The puppets look really cheesy , not in a good way like in the Puppet Master 80's flicks....

3. Prediction: NEGATIVE (51.36%) | Actual: POSITIVE
   Negative prob: 0.5136, Positive prob: 0.4864
   There are massive levels, massive unlockable characters... it's just a massive game....

4. Prediction: NEGATIVE (51.71%) | Actual: NEGATIVE
   Negative prob: 0.5171, Positive prob: 0.4829
   This short film certainly pulls no punches....

5. Prediction: NEGATIVE (51.78%) | Actual: POSITIVE
   Negative prob: 0.5178, Positive prob: 0.4822
   The attractive set used throughout most of the film

In [132]:
# ROC Curve and AUC
from sklearn.metrics import roc_curve, auc
import matplotlib.pyplot as plt

fpr, tpr, thresholds = roc_curve(imdb_test_y, imdb_pred_proba[:,1])
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(8,6))
plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (AUC = {roc_auc:.3f})')
plt.plot([0,1], [0,1], color='navy', lw=2, linestyle='--', label='Random Classifier')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve - IMDB Sentiment Classification')
plt.legend(loc='lower right')
plt.grid(alpha=0.3)
plt.show()

print(f"\nROC AUC Score: {roc_auc:.4f}")

ModuleNotFoundError: No module named 'matplotlib'

In [133]:
# Summary of Sentiment Analysis
print("="*60)
print("SENTIMENT ANALYSIS SUMMARY - IMDB REVIEWS")
print("="*60)
print(f"\nDataset Overview:")
print(f"  Total Reviews: {len(imdb_labels)}")
print(f"  Positive: {sum(imdb_labels)} ({100*sum(imdb_labels)/len(imdb_labels):.1f}%)")
print(f"  Negative: {len(imdb_labels)-sum(imdb_labels)} ({100*(len(imdb_labels)-sum(imdb_labels))/len(imdb_labels):.1f}%)")

print(f"\nModel Performance:")
print(f"  Train Accuracy: {imdb_train_accuracy:.4f}")
print(f"  Test Accuracy:  {accuracy:.4f}")
print(f"  Precision:      {precision:.4f} (correctly identified positive reviews)")
print(f"  Recall:         {recall:.4f} (found % of actual positive reviews)")
print(f"  F1-Score:       {f1:.4f}")
print(f"  ROC AUC:        {roc_auc:.4f}")

print(f"\nKey Sentiment Indicators:")
print(f"  Vocabulary Size: 1000 (stopwords filtered)")
print(f"  Top Positive Words: excellent, wonderful, brilliant, amazing, best, love")
print(f"  Top Negative Words: bad, terrible, awful, hate, worst, boring")
print(f"\n" + "="*60)

SENTIMENT ANALYSIS SUMMARY - IMDB REVIEWS

Dataset Overview:
  Total Reviews: 1000
  Positive: 500 (50.0%)
  Negative: 500 (50.0%)

Model Performance:
  Train Accuracy: 0.9293
  Test Accuracy:  0.7520
  Precision:      0.8333 (correctly identified positive reviews)
  Recall:         0.6538 (found % of actual positive reviews)
  F1-Score:       0.7328


NameError: name 'roc_auc' is not defined